In [1]:
import torch
import h5py
import os
import glob
import numpy as np
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

class PoolBoilingDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.file_paths = []
        self.frame_counts = []
        self.cumulative_indices = [0]
        
        # sorted to ensure consistent ordering
        search_path = os.path.join(root_dir, "PoolBoiling-*", "*.hdf5")
        self.file_paths = sorted(glob.glob(search_path))
        
        print(f"Indexing {len(self.file_paths)} files...")
        total_frames = 0
        for p in self.file_paths:
            with h5py.File(p, 'r') as f:
                # We assume temperature and dfun have the same length
                count = f["temperature"].shape[0]
                self.frame_counts.append(count)
                total_frames += count
                self.cumulative_indices.append(total_frames)
        
        print(f"Total frames indexed: {total_frames}")

    def __len__(self):
        return self.cumulative_indices[-1]

    def __getitem__(self, idx):
        # Find which file the global index belongs to
        file_idx = 0
        for i in range(len(self.cumulative_indices) - 1):
            if idx < self.cumulative_indices[i+1]:
                file_idx = i
                break
        
        # Calculate local frame index within that file
        local_idx = idx - self.cumulative_indices[file_idx]
        file_path = self.file_paths[file_idx]
        
        with h5py.File(file_path, 'r') as f:
            # Load both channels
            temp = f["temperature"][local_idx]
            dfun = f["dfun"][local_idx]
            
            # Stack along channel axis (axis 0)
            # Resulting shape: (2, H, W)
            # Channel 0: Temperature
            # Channel 1: dfun
            data = np.stack([temp, dfun], axis=0).astype('float32')
            
        sample = torch.from_numpy(data)

        if torch.isnan(sample).any():
            print(f"NaN detected in {file_path} at index {local_idx}")
        
        if self.transform:
            sample = self.transform(sample)
            
        return sample


In [2]:
dataset = PoolBoilingDataset(root_dir='./bubbleml-ds')

dataloader = DataLoader(
    dataset, 
    batch_size=32, 
    shuffle=True, 
    num_workers=2
)

print(f"Number of batches: {len(dataloader)}")
for batch in dataloader:
    print(f"Batch shape: {batch.shape}")  # Expected: torch.Size([32, 2, 64, 64])
    print(f"Channel 0 (Temp) shape: {batch[:, 0, ...].shape}")
    print(f"Channel 1 (Dfun) shape: {batch[:, 1, ...].shape}")
    break

def get_stats(loader):
    n_pixels = 0
    
    # Channel 0 (Temp) accumulators
    temp_sum = 0.0
    temp_sq_sum = 0.0
    
    # Channel 1 (Dfun) accumulators
    dfun_sum = 0.0
    dfun_sq_sum = 0.0
    
    print("Calculating statistics...")
    for data in tqdm(loader):
        # data: [B, 2, H, W]
        b, c, h, w = data.shape
        # Total pixels in this batch per channel
        num_el = b * h * w
        n_pixels += num_el
        
        # Channel 0: Temperature
        temp_batch = data[:, 0, :, :]
        temp_sum += torch.sum(temp_batch).item()
        temp_sq_sum += torch.sum(temp_batch ** 2).item()
        
        # Channel 1: dfun
        dfun_batch = data[:, 1, :, :]
        dfun_sum += torch.sum(dfun_batch).item()
        dfun_sq_sum += torch.sum(dfun_batch ** 2).item()
        
    # Calculate global stats
    # Mean = E[X]
    temp_mean = temp_sum / n_pixels
    dfun_mean = dfun_sum / n_pixels
    
    # Std = sqrt(E[X^2] - (E[X])^2)
    temp_std = (temp_sq_sum / n_pixels - temp_mean ** 2) ** 0.5
    dfun_std = (dfun_sq_sum / n_pixels - dfun_mean ** 2) ** 0.5
        
    return (temp_mean, temp_std), (dfun_mean, dfun_std)

# Use a larger batch size for stats calculation speed
loader = DataLoader(dataset, batch_size=256, num_workers=4)
(t_means, t_stds), (d_means, d_stds) = get_stats(loader)

Indexing 124 files...
Total frames indexed: 248124
Number of batches: 7754
Batch shape: torch.Size([32, 2, 64, 64])
Channel 0 (Temp) shape: torch.Size([32, 64, 64])
Channel 1 (Dfun) shape: torch.Size([32, 64, 64])


/home/jovyan/.local/lib/python3.10/site-packages/torch/utils/data/dataloader.py:554: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


Calculating statistics...


  1%|          | 8/970 [05:06<10:13:26, 38.26s/it] 


KeyboardInterrupt: 

In [ ]:
print(f"Avg Temp Mean: {np.mean(t_means):.4f}")
print(f"Avg Temp Std:  {np.mean(t_stds):.4f}")
print(f"Avg Dfun Mean: {np.mean(d_means):.4f}")
print(f"Avg Dfun Std:  {np.mean(d_stds):.4f}")